In [ ]:
!pip install -q groq
!pip install -q openai
!pip install -q vllm
!pip install -q sacrebleu rouge-score
!pip install -q langchain langchain-community langchain-qdrant langchain-huggingface
!pip install -q langchain-text-splitters qdrant-client sentence-transformers rank-bm25
!pip install -q -U langchain-groq langchain-openai

In [9]:
import json
import time
import sacrebleu
import re
import os
import pandas as pd
from pprint import pprint
from groq import Groq
from google.colab import userdata
from openai import OpenAI
from tqdm.auto import tqdm
from rouge_score import rouge_scorer
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from qdrant_client import QdrantClient
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 1. EDA

In [6]:
with open("articles.json", "r", encoding="utf-8") as f:
    articles = json.load(f)

with open("questions.json", "r", encoding="utf-8") as f:
    questions = json.load(f)

with open("ground_truth.json", "r", encoding="utf-8") as f:
    ground_truth = json.load(f)

In [ ]:
def analyze_data(data, name):
    print("=" * 60)
    print(name)
    print("=" * 60)

    print("Type:", type(data))

    if isinstance(data, list):
        print("Number of elements:", len(data))

        if len(data) > 0:
            print("First element type:", type(data[0]))

            if isinstance(data[0], dict):
                print("Keys:", list(data[0].keys()))

                print("\nFirst element:")
                pprint(data[0])

    elif isinstance(data, dict):
        print("Number of keys:", len(data))
        print("Keys:", list(data.keys()))

        print("\nFirst elements:")
        for i, (key, value) in enumerate(data.items()):
            print(f"\nKey: {key}")
            pprint(value)

            if i >= 2:
                break


analyze_data(articles, "ARTICLES")
analyze_data(questions, "QUESTIONS")
analyze_data(ground_truth, "GROUND TRUTH")

Имеем 10 статей, 50 вопросов и 50 ответов на них

In [7]:
articles_df = pd.DataFrame(articles)
questions_df = pd.DataFrame(questions)
ground_truth_df = pd.DataFrame(ground_truth)

In [ ]:
assert set(questions_df["id"]) == set(ground_truth_df["id"])

assert questions_df.isna().sum().sum() == 0
assert ground_truth_df.isna().sum().sum() == 0

print("Dataset validation passed!")

Пропусков среди вопросов и ответов нет.

In [ ]:
test_df = questions_df.merge(
    ground_truth_df,
    on="id"
)

display(test_df.head())

# 2. Zero-shot

## 2.1 Qwen3.8-27b по API

In [ ]:
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
client = Groq(api_key=GROQ_API_KEY)
MODEL_NAME = "qwen/qwen3.8-27b"

def generate_answer(question):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": question
            }
        ],
        max_completion_tokens=300,
        temperature=0,
        reasoning_effort="none"
    )

    return response.choices[0].message.content.strip()

baseline_answers = []

for i, question in enumerate(test_df["question"], start=1):
    print(f"Processing {i}/{len(test_df)}")

    answer = generate_answer(question)
    baseline_answers.append(answer)


baseline_df = test_df[
    ["id", "question", "answer"]
].copy()

baseline_df["baseline_answer"] = baseline_answers

baseline_df.to_csv(
    "baseline_groq_qwen.csv",
    index=False,
    encoding="utf-8-sig"
)

baseline_df.to_json(
    "baseline_groq_qwen.json",
    orient="records",
    force_ascii=False,
    indent=2
)

print("\nDone!")
print(f"Model: {MODEL_NAME}")
print(f"Questions: {len(baseline_df)}")

display(baseline_df.head(10))

## 2.2 Local Llama-3.2-3B-Instruct

In [ ]:
client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY"
)

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

def generate_answer(question):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": question
            }
        ],
        temperature=0,
        max_tokens=300
    )

    return response.choices[0].message.content.strip()

local_answers = []

for i, question in enumerate(test_df["question"], start=1):
    print(f"Processing {i}/{len(test_df)}")

    answer = generate_answer(question)
    local_answers.append(answer)

baseline_llama_df = test_df[["id", "question", "answer"]].copy()
baseline_llama_df["baseline_answer"] = local_answers

baseline_llama_df.to_csv(
    "baseline_llama_3b.csv",
    index=False,
    encoding="utf-8-sig"
)

baseline_llama_df.to_json(
    "baseline_llama_3b.json",
    orient="records",
    force_ascii=False,
    indent=2
)

print("Saved:")
print("baseline_llama_3b.csv")
print("baseline_llama_3b.json")

In [ ]:
with pd.option_context(
    "display.max_colwidth", None,
    "display.max_columns", None,
    "display.width", None
):
    display(baseline_llama_df.head(10))

## 2.3 Оценка baseline

In [16]:
def calculate_metrics(predictions: list, references: list) -> dict:
    """Расчёт метрик BLEU и ROUGE-1/2/L."""
    bleu = sacrebleu.corpus_bleu(predictions, [references])

    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"],
        use_stemmer=True
    )

    rouge_scores = [
        scorer.score(ref, pred)
        for pred, ref in zip(predictions, references)
    ]

    r1 = sum(
        s["rouge1"].fmeasure for s in rouge_scores
    ) / len(rouge_scores)

    r2 = sum(
        s["rouge2"].fmeasure for s in rouge_scores
    ) / len(rouge_scores)

    rL = sum(
        s["rougeL"].fmeasure for s in rouge_scores
    ) / len(rouge_scores)

    return {
        "BLEU": bleu.score,
        "ROUGE-1": r1,
        "ROUGE-2": r2,
        "ROUGE-L": rL,
    }

In [ ]:
groq_df = pd.read_csv("baseline_groq_qwen.csv")
llama_df = pd.read_csv("baseline_llama_3b.csv")

groq_predictions = groq_df["baseline_answer"].fillna("").tolist()
groq_references = groq_df["answer"].fillna("").tolist()

groq_metrics = calculate_metrics(
    groq_predictions,
    groq_references
)

llama_predictions = llama_df["baseline_answer"].fillna("").tolist()
llama_references = llama_df["answer"].fillna("").tolist()

llama_metrics = calculate_metrics(
    llama_predictions,
    llama_references
)


metrics_df = pd.DataFrame([
    {
        "Model": "Qwen 3.8 27B (Groq)",
        **groq_metrics
    },
    {
        "Model": "Llama 3.2 3B Instruct",
        **llama_metrics
    }
])

display(
    metrics_df.style.format({
        "BLEU": "{:.2f}",
        "ROUGE-1": "{:.4f}",
        "ROUGE-2": "{:.4f}",
        "ROUGE-L": "{:.4f}"
    })
)


metrics_df.to_csv(
    "baseline_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
client = Groq(api_key=GROQ_API_KEY)
JUDGE_MODEL = "qwen/qwen3.8-27b"


def judge_answer(question, answer, ground_truth):

    prompt = f"""
Ты являешься оценщиком систем вопросно-ответного поиска.

Оцени ответ модели, сравнив его с эталонным ответом.

Вопрос:

{question}

Ответ модели:

{answer}

Эталонный ответ:

{ground_truth}

Используй следующую шкалу оценки:

2 = полностью корректный ответ: ответ содержит правильную информацию
и не содержит существенных фактических ошибок.

1 = частично корректный ответ: ответ содержит часть правильной информации,
но является неполным, слишком расплывчатым или содержит незначительную
фактическую ошибку.

0 = некорректный ответ: ответ неправильный, не относится к вопросу
или содержит существенную фактическую ошибку.

Оценивай прежде всего фактическую корректность ответа.
Не требуй дословного совпадения с эталонным ответом.
Если ответ сформулирован другими словами, но передаёт ту же информацию,
считай его корректным.

Верни ТОЛЬКО JSON-объект строго в следующем формате:

{{"score": 0}}

Не добавляй никаких пояснений.
"""

    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        max_completion_tokens=50,
        reasoning_effort="none"
    )

    text = response.choices[0].message.content.strip()

    try:
        result = json.loads(text)
        score = int(result["score"])

        if score not in [0, 1, 2]:
            raise ValueError

        return score

    except (json.JSONDecodeError, KeyError, ValueError):
        print("Invalid judge response:", text)
        return None


groq_df = pd.read_csv("baseline_groq_qwen.csv")
llama_df = pd.read_csv("baseline_llama_3b.csv")

groq_judge_scores = []

for i, row in groq_df.iterrows():
    print(f"Groq: {i + 1}/{len(groq_df)}")

    score = judge_answer(
        question=row["question"],
        answer=row["baseline_answer"],
        ground_truth=row["answer"]
    )

    groq_judge_scores.append(score)

    time.sleep(0.2)


llama_judge_scores = []

for i, row in llama_df.iterrows():
    print(f"Llama: {i + 1}/{len(llama_df)}")

    score = judge_answer(
        question=row["question"],
        answer=row["baseline_answer"],
        ground_truth=row["answer"]
    )

    llama_judge_scores.append(score)

    time.sleep(0.2)

groq_df["judge_score"] = groq_judge_scores
llama_df["judge_score"] = llama_judge_scores


judge_results = pd.DataFrame([
    {
        "Model": "Qwen 3.8 27B (Groq)",
        "LLM-as-a-Judge": groq_df["judge_score"].mean()
    },
    {
        "Model": "Llama 3.2 3B Instruct",
        "LLM-as-a-Judge": llama_df["judge_score"].mean()
    }
])


display(
    judge_results.style.format({
        "LLM-as-a-Judge": "{:.3f}"
    })
)

groq_df.to_csv(
    "baseline_groq_qwen_judge.csv",
    index=False,
    encoding="utf-8-sig"
)

llama_df.to_csv(
    "baseline_llama_3b_judge.csv",
    index=False,
    encoding="utf-8-sig"
)

judge_results.to_csv(
    "baseline_judge_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
print("Qwen:")
print(groq_df["judge_score"].value_counts().sort_index())

print("\nLlama:")
print(llama_df["judge_score"].value_counts().sort_index())

# 3. RAG

## 3.1 RAG with large LLM

In [ ]:
with open("articles.json", "r", encoding="utf-8") as f:
    articles = json.load(f)

documents = [
    Document(
        page_content=article["text"],
        metadata={
            "id": article["id"],
            "title": article["title"]
        }
    )
    for article in articles
]


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(f"Количество статей: {len(documents)}")
print(f"Количество chunks: {len(chunks)}")


embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={
        "device": "cuda"
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

print("BGE-M3 загружена")



vector_store = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embedding_model,
    location=":memory:",
    collection_name="corporate_kb"
)

print("Qdrant collection создана")


dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10}
)


bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 10


hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, bm25_retriever],
    weights=[0.5, 0.5]
)

print("Hybrid retriever готов.")

In [ ]:
question = "Как изменился средний чек новых контрактов по сравнению с Q4 2025?"

results = hybrid_retriever.invoke(question)


for i, document in enumerate(results[:3], 1):
    print(f"===== CHUNK {i} =====")
    print("Article:", document.metadata["title"])
    print("Article ID:", document.metadata["id"])
    print(document.page_content)
    print()


In [ ]:
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

MODEL_NAME = "qwen/qwen3.8-27b"

llm = ChatGroq(
    model=MODEL_NAME,
    temperature=0,
    max_tokens=300,
)

print(f"Модель: {MODEL_NAME}")

prompt = ChatPromptTemplate.from_template("""
Ты отвечаешь на вопросы по внутренней базе знаний компании.

Используй только информацию из контекста. Не придумывай факты и не используй
внешние знания.

Ответ должен быть кратким, самодостаточным и естественным. Сформулируй
полное предложение, а не просто число, дату или короткую фразу.
Отвечай только на заданный вопрос и не пересказывай лишнюю информацию
из контекста. Если вопрос требует нескольких значений, укажи их все.

Если ответа действительно нет в контексте, напиши:
"Информация не найдена в базе знаний."

Контекст:
{context}

Вопрос:
{question}

Ответ:
""")


generation_chain = (prompt | llm | StrOutputParser())

print("Generation chain готов.")

def format_docs(docs):
    return "\n\n".join(
        [
            f"[Источник {i}] "
            f"{doc.metadata['title']} "
            f"(ID статьи: {doc.metadata['id']})\n"
            f"{doc.page_content}"
            for i, doc in enumerate(docs, 1)
        ]
    )


rag_results = []

for _, row in tqdm(
    test_df.iterrows(),
    total=len(test_df),
    desc="Generating RAG answers"
):

    question = row["question"]
    documents = hybrid_retriever.invoke(question)
    documents = documents[:3]
    context = format_docs(documents)

    answer = generation_chain.invoke({
        "context": context,
        "question": question
    })

    rag_results.append({
        "id": row["id"],
        "question": question,
        "answer": row["answer"],
        "rag_answer": answer,
        "retrieved_context": context
    })


rag_df = pd.DataFrame(rag_results)

print(f"\nГотово: {len(rag_df)} ответов")

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)

display(
    rag_df[
        [
            "id",
            "question",
            "answer",
            "rag_answer"
        ]
    ]
)


rag_df.to_csv(
    "rag_qwen_results.csv",
    index=False,
    encoding="utf-8-sig"
)


print("\nФайлы сохранены:")
print("rag_qwen_results.csv")

## 3.2 RAG with small LLM

In [ ]:
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

llm_local = ChatOpenAI(
    model=MODEL_NAME,
    base_url="http://localhost:8000/v1",
    api_key="EMPTY",
    temperature=0,
    max_tokens=300,
)

print(f"Модель: {MODEL_NAME}")
print("vLLM подключён")

prompt_local = ChatPromptTemplate.from_template("""
Ответь на вопрос, используя только информацию из контекста.

Правила:
- Не придумывай информацию и не используй внешние знания.
- Ответь непосредственно на вопрос.
- Сформулируй самодостаточный ответ: не отвечай только числом, датой или короткой фразой.
- Если вопрос содержит несколько частей, ответь на каждую из них.
- Не добавляй информацию, которая не нужна для ответа.
- Не упоминай источники, номера источников или контекст.
- Если ответа на вопрос действительно нет в контексте, напиши:
"Информация не найдена в базе знаний."

Контекст:
{context}

Вопрос:
{question}

Ответ:
""")

generation_chain_local = (prompt_local | llm_local | StrOutputParser())

print("Generation chain готов.")

In [ ]:
rag_local_results = []

for _, row in tqdm(
    test_df.iterrows(),
    total=len(test_df),
    desc="Generating Local Llama RAG answers"
):

    question = row["question"]

    documents = hybrid_retriever.invoke(question)[:3]

    context = format_docs(documents)

    answer = generation_chain_local.invoke({
        "context": context,
        "question": question
    })

    rag_local_results.append({
        "id": row["id"],
        "question": question,
        "answer": row["answer"],
        "rag_answer": answer,
        "retrieved_context": context
    })


rag_local_df = pd.DataFrame(rag_local_results)

print(f"\nГотово: {len(rag_local_df)} ответов")


pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)

display(
    rag_local_df[
        [
            "id",
            "question",
            "answer",
            "rag_answer"
        ]
    ]
)


rag_local_df.to_csv(
    "rag_llama_3b_vllm_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nФайлы сохранены:")
print("rag_llama_3b_vllm_results.csv")

## 3.3 Сравнение RAG

In [ ]:
groq_df = pd.read_csv("rag_qwen_results.csv")
llama_df = pd.read_csv("rag_llama_3b_vllm_results.csv")

groq_predictions = groq_df["rag_answer"].fillna("").tolist()
groq_references = groq_df["answer"].fillna("").tolist()

groq_metrics = calculate_metrics(
    groq_predictions,
    groq_references
)

llama_predictions = llama_df["rag_answer"].fillna("").tolist()
llama_references = llama_df["answer"].fillna("").tolist()

llama_metrics = calculate_metrics(
    llama_predictions,
    llama_references
)


metrics_df = pd.DataFrame([
    {
        "RAG with": "Qwen 3.8 27B (Groq)",
        **groq_metrics
    },
    {
        "RAG with": "Llama 3.2 3B Instruct",
        **llama_metrics
    }
])

display(
    metrics_df.style.format({
        "BLEU": "{:.2f}",
        "ROUGE-1": "{:.4f}",
        "ROUGE-2": "{:.4f}",
        "ROUGE-L": "{:.4f}"
    })
)


metrics_df.to_csv(
    "RAG_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
client = Groq(api_key=GROQ_API_KEY)
JUDGE_MODEL = "qwen/qwen3.8-27b"


def judge_answer(question, answer, ground_truth):

    prompt = f"""
Оцени, правильно ли ответ модели отвечает на вопрос.

Вопрос:
{question}

Ответ модели:
{answer}

Эталон:
{ground_truth}

2 — полностью правильный ответ.
1 — частично правильный или неполный ответ.
0 — неправильный ответ.

Оценивай смысл, а не дословное совпадение.
Краткая или иначе сформулированная версия правильного ответа = 2.
Дополнительная корректная информация не является ошибкой.
Если ответ содержит правильное значение, но не повторяет очевидную из вопроса
единицу, период или контекст, ставь 2.

Верни только JSON:
{{"score": 0}}
"""


    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        max_completion_tokens=50,
        reasoning_effort="none"
    )

    text = response.choices[0].message.content.strip()

    try:
        result = json.loads(text)
        score = int(result["score"])

        if score not in [0, 1, 2]:
            raise ValueError

        return score

    except (json.JSONDecodeError, KeyError, ValueError):
        print("Invalid judge response:", text)
        return None

groq_df = pd.read_csv("rag_qwen_results.csv")
llama_df = pd.read_csv("rag_llama_3b_vllm_results.csv")

groq_judge_scores = []

for i, row in groq_df.iterrows():
    print(f"Groq: {i + 1}/{len(groq_df)}")

    score = judge_answer(
        question=row["question"],
        answer=row["rag_answer"],
        ground_truth=row["answer"]
    )

    groq_judge_scores.append(score)

    time.sleep(0.2)


llama_judge_scores = []

for i, row in llama_df.iterrows():
    print(f"Llama: {i + 1}/{len(llama_df)}")

    score = judge_answer(
        question=row["question"],
        answer=row["rag_answer"],
        ground_truth=row["answer"]
    )

    llama_judge_scores.append(score)

    time.sleep(0.2)

groq_df["judge_score"] = groq_judge_scores
llama_df["judge_score"] = llama_judge_scores


judge_results = pd.DataFrame([
    {
        "RAG with": "Qwen 3.8 27B (Groq)",
        "LLM-as-a-Judge": groq_df["judge_score"].mean()
    },
    {
        "RAG with": "Llama 3.2 3B Instruct",
        "LLM-as-a-Judge": llama_df["judge_score"].mean()
    }
])


display(
    judge_results.style.format({
        "LLM-as-a-Judge": "{:.3f}"
    })
)

groq_df.to_csv(
    "rag_groq_qwen_judge.csv",
    index=False,
    encoding="utf-8-sig"
)

llama_df.to_csv(
    "rag_llama_3b_judge.csv",
    index=False,
    encoding="utf-8-sig"
)

judge_results.to_csv(
    "rag_judge_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
print("Qwen:")
print(groq_df["judge_score"].value_counts().sort_index())

print("\nLlama:")
print(llama_df["judge_score"].value_counts().sort_index())

In [ ]:
low_score = llama_df[llama_df["judge_score"].isin([0, 1])]

display(
    low_score[
        ["id", "question", "answer", "rag_answer", "judge_score"]
    ]
)